In [ ]:
# Kaggle setup (does nothing outside Kaggle). Needs: GPU on, Internet on, competition data as input.
# The repo stays private until the deadline, so it is cloned with a GitHub token saved as a
# Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets; fine-grained token, read-only, this repo only).
import os, subprocess
REPO = "polluvl/hw2-video-detection"   # <- set once, e.g. "polluvl/hw2-video-detection"
if os.path.isdir("/kaggle/working") and not os.path.isdir("/kaggle/working/repo"):
    try:
        from kaggle_secrets import UserSecretsClient
        auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
    except Exception:
        auth = ""  # no secret -> works once the repo is public
    r = subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
                       capture_output=True)
    if r.returncode != 0:  # do not print the command: it contains the token
        raise RuntimeError("git clone failed: check REPO and the GITHUB_TOKEN secret")
    # remove the token from the saved clone (outputs of /kaggle/working can be shared)
    subprocess.run(["git", "-C", "/kaggle/working/repo", "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)
    subprocess.run(["pip", "install", "-q", "-r", "/kaggle/working/repo/requirements.txt"], check=True)
if os.path.isdir("/kaggle/working/repo/notebooks"):
    os.chdir("/kaggle/working/repo/notebooks")

# 02 · Two-stage anchor-based: Faster R-CNN (torchvision, ResNet-50-FPN v2, COCO-pretrained)

**Fixed protocol (same for 03 RetinaNet):**
train = every `TRAIN_STEP`-th frame of the 56 train clips, val = every `VAL_STEP`-th frame of
the 7 val clips (split by sequence, given by the organisers). Images resized to short side 800 /
long side ≤ 1333 inside the model, horizontal flip + brightness/contrast jitter, ignored regions
painted with the mean colour during training. SGD (momentum 0.9, wd 1e-4), lr 0.01, batch 4,
300-iteration warmup + cosine, AMP, `EPOCHS` epochs, best epoch by val mAP.
Evaluation: COCO mAP@[.5:.95], AP50, AP_small with maxDets=500 and ignored regions as iscrowd.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("src").resolve()))
from analysis import (draw_errors, error_breakdown, recall_by_size,
                      summarise_errors, worst_frames)
from common import (SMOKE, evaluate, evaluate_per_clip, find_data_root,
                    list_frames, log_result, save_preds, seed_everything, work_dir)
from tv_detection import build_faster_rcnn, predict, train_detector

DATA = find_data_root()
OUT = work_dir()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_STEP, VAL_STEP, EPOCHS = (1, 1, 1) if SMOKE else (10, 5, 8)
train_frames = list_frames(DATA / "train", step=TRAIN_STEP)
val_frames = list_frames(DATA / "val", step=VAL_STEP)
print(f"device {DEVICE} | train {len(train_frames)} frames | val {len(val_frames)} frames")

BASE = dict(epochs=EPOCHS, batch_size=2 if SMOKE else 4, lr=0.01, num_workers=2,
            min_size=800, max_size=1333)
if SMOKE:
    BASE.update(min_size=192, max_size=320, print_every=5)

## Experiments
Each run changes ONE thing relative to the baseline:
- `frcnn_base` – torchvision defaults (Smooth-L1 box loss, 512 RoIs/image, 25 % positives)
- `frcnn_giou` – loss modification: GIoU loss for the second-stage box regression
- `frcnn_roi`  – RoI sampling: 256 RoIs with 50 % positives (many small objects per image →
  more positive RoIs per GT)

The number of RPN proposals at test time is checked afterwards **without retraining** (cheap).

In [ ]:
RUNS = {
    "frcnn_base": {},
    "frcnn_giou": {"box_loss": "giou"},
    "frcnn_roi": {"roi_batch_size": 256, "roi_pos_fraction": 0.5},
}

histories = {}
for name, changes in RUNS.items():
    seed_everything(42)
    cfg = {**BASE, **changes}
    print(f"\n=== {name}: {changes}")
    model = build_faster_rcnn(cfg, pretrained=not SMOKE)
    histories[name] = train_detector(model, train_frames, val_frames, cfg, DEVICE, OUT / f"{name}.pt")

    preds, ms = predict(model, val_frames, DEVICE)
    save_preds(preds, OUT / f"preds_val_{name}.npz")
    metrics = evaluate(val_frames, preds)
    log_result(name, metrics, {"family": "two-stage", "ms_per_img": round(ms, 1),
                               "train_min": round(sum(h["train_min"] for h in histories[name]), 1),
                               "config": str(changes)})
    print(name, metrics, f"{ms:.1f} ms/img")
    del model
    torch.cuda.empty_cache()

## Learning curves and results table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name, h in histories.items():
    h = pd.DataFrame(h)
    axes[0].plot(h.epoch, h.mAP, marker="o", label=name)
    axes[1].plot(h.epoch, h.AP_small, marker="o", label=name)
axes[0].set_title("val mAP@[.5:.95]")
axes[1].set_title("val AP_small")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(OUT / "figures_frcnn_curves.png", dpi=120)
plt.show()

results = pd.read_csv(OUT / "results.csv")
results[results.experiment.str.startswith("frcnn")]

## Number of RPN proposals at test time (no retraining)
Dense frames can contain hundreds of objects; with too few proposals some objects never
reach the second stage.

In [ ]:
best_name = results[results.experiment.isin(RUNS)].sort_values("mAP").experiment.iloc[-1]
print("best run:", best_name)
model = build_faster_rcnn({**BASE, **RUNS[best_name]}, pretrained=False)
model.load_state_dict(torch.load(OUT / f"{best_name}.pt", map_location=DEVICE))
model.to(DEVICE)

sweep = []
for n_prop in [300, 1000, 2000, 3000]:
    model.rpn._pre_nms_top_n["testing"] = max(n_prop, 1000)
    model.rpn._post_nms_top_n["testing"] = n_prop
    p, ms = predict(model, val_frames, DEVICE)
    sweep.append({"proposals": n_prop, **evaluate(val_frames, p), "ms_per_img": round(ms, 1)})
sweep = pd.DataFrame(sweep)
print(sweep[["proposals", "mAP", "AP50", "AP_small", "AR", "ms_per_img"]])
model.rpn._pre_nms_top_n["testing"] = 1000
model.rpn._post_nms_top_n["testing"] = 1000

## What does each stage contribute?
Recall of the **RPN proposals** (class-agnostic, before the second stage) versus the **final
detections**, at IoU 0.5 and 0.7, by object size. The RPN decides which objects can be found at
all; the second stage classifies them and refines the box.

In [ ]:
from analysis import SIZE_BINS, SIZE_LABELS, box_iou
from common import read_labels
from tv_detection import DroneDataset, collate
from torch.utils.data import DataLoader


@torch.no_grad()
def rpn_proposals(model, frames):
    model.eval()
    out = {}
    loader = DataLoader(DroneDataset(frames), batch_size=2, collate_fn=collate)
    for images, _, keys in loader:
        orig_sizes = [img.shape[-2:] for img in images]
        image_list, _ = model.transform([img.to(DEVICE) for img in images])
        features = model.backbone(image_list.tensors)
        proposals, _ = model.rpn(image_list, features)
        for key, props, (h0, w0), (h1, w1) in zip(keys, proposals, orig_sizes, image_list.image_sizes):
            scale = torch.tensor([w0 / w1, h0 / h1, w0 / w1, h0 / h1], device=props.device)
            out[key] = (props * scale).cpu().numpy()
    return out


def recall_table(frames, boxes_by_key, iou_thr, score_thr=None):
    hits, sizes = [], []
    for r in frames.itertuples():
        gt, _ = read_labels(r.label_path, r.width, r.height)
        b = boxes_by_key.get(r.image_key, np.zeros((0, 6)))
        if score_thr is not None and len(b):
            b = b[b[:, 4] >= score_thr]
        iou = box_iou(gt, b[:, :4]) if len(b) else np.zeros((len(gt), 0))
        hits += list(iou.max(axis=1) >= iou_thr) if iou.shape[1] else [False] * len(gt)
        sizes += list(np.sqrt((gt[:, 2] - gt[:, 0]) * (gt[:, 3] - gt[:, 1])))
    df = pd.DataFrame({"hit": hits, "size_bin": pd.cut(sizes, SIZE_BINS, labels=SIZE_LABELS)})
    return df.groupby("size_bin", observed=False).hit.mean().round(3)


props = rpn_proposals(model, val_frames)
best_preds, _ = predict(model, val_frames, DEVICE)
stage_table = pd.DataFrame({
    "RPN proposals (1000) IoU≥0.5": recall_table(val_frames, props, 0.5),
    "RPN proposals IoU≥0.7": recall_table(val_frames, props, 0.7),
    "final dets (score≥0.05) IoU≥0.5": recall_table(val_frames, best_preds, 0.5, 0.05),
    "final dets IoU≥0.7": recall_table(val_frames, best_preds, 0.7, 0.05),
})
stage_table
# (final-detection recall here is class-agnostic too, so it is directly comparable to the RPN)

## Failure cases (score threshold 0.3)

In [ ]:
gts, fps = error_breakdown(val_frames, best_preds, score_thr=0.3)
by_size, by_class, fp_types = summarise_errors(gts, fps)
print("GT outcome by size:\n", by_size, "\n\nGT outcome by class:\n", by_class, "\n\nFalse positives:\n", fp_types)
print("\nper-clip mAP:\n", evaluate_per_clip(val_frames, best_preds))

worst = worst_frames(gts, val_frames, k=3)
fig, axes = plt.subplots(len(worst), 1, figsize=(14, 8 * len(worst)))
for ax, row in zip(np.atleast_1d(axes), worst.itertuples()):
    draw_errors(ax, row, best_preds[row.image_key], score_thr=0.3)
plt.tight_layout()
plt.savefig(OUT / "figures_frcnn_failures.png", dpi=100)
plt.show()

TODO (your analysis): for each of the 3 frames above, describe what goes wrong
(green = found, red = missed, orange = wrong class / bad box, yellow dashed = false positive)
and give a plausible cause, linked to the tables above (size, class, density, lighting...).